# RouterSense Q1 Revised Full Solution

This notebook:
- loads the given iOS and Android JSON files
- flattens them into tabular data
- combines both stores into one union dataset
- adds all required Q1 fields
- produces:
  - `q1_augmented_full_union.csv`
  - `q1_submission_draft.csv`

It is designed to save files in the **current working directory**.


In [1]:
import json
import re
import os
from pathlib import Path
import pandas as pd

APP_STORE_JSON = Path("app_store_apps_details.json")
PLAY_STORE_JSON = Path("google_play_apps_details.json")

assert APP_STORE_JSON.exists(), f"Missing file: {APP_STORE_JSON.resolve()}"
assert PLAY_STORE_JSON.exists(), f"Missing file: {PLAY_STORE_JSON.resolve()}"

print("Working directory:", os.getcwd())
print("Found:", APP_STORE_JSON.resolve())
print("Found:", PLAY_STORE_JSON.resolve())


Working directory: /Users/naman/Downloads/RouterSense-tasks/Q1)
Found: /Users/naman/Downloads/RouterSense-tasks/Q1)/app_store_apps_details.json
Found: /Users/naman/Downloads/RouterSense-tasks/Q1)/google_play_apps_details.json


In [2]:
with open(APP_STORE_JSON, "r", encoding="utf-8") as f:
    ios_raw = json.load(f)

with open(PLAY_STORE_JSON, "r", encoding="utf-8") as f:
    android_raw = json.load(f)

ios_results = ios_raw.get("results", [])
android_results = android_raw.get("results", [])

print("iOS rows:", len(ios_results))
print("Android rows:", len(android_results))


iOS rows: 453
Android rows: 492


## Helper functions


In [3]:
def safe_join(value, sep=", "):
    if value is None:
        return ""
    if isinstance(value, list):
        return sep.join(str(x) for x in value)
    return str(value)

def normalize_lang_codes(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(x).lower() for x in value)
    return str(value).lower()

def first_nonempty(*values):
    for v in values:
        if v is None:
            continue
        if isinstance(v, str) and v.strip() == "":
            continue
        return v
    return ""

def contains_any(text, keywords):
    text = (text or "").lower()
    return any(k in text for k in keywords)

companion_keywords = [
    "boyfriend", "girlfriend", "husband", "wife", "lover", "romance", "romantic",
    "companion", "partner", "virtual friend", "ai friend", "friend", "soulmate",
    "dating", "roleplay", "role-play", "emotional support", "relationship",
    "intimate", "flirty", "flirt", "character", "characters", "celebrity",
    "virtual boyfriend", "virtual girlfriend"
]

general_keywords = [
    "assistant", "productivity", "homework", "study", "writing", "idea generation",
    "creative workflow", "multimodal", "search", "copilot", "chatgpt", "claude",
    "gemini", "deepseek", "perplexity", "general ai", "broad assistant"
]

mixed_keywords = [
    "companion and assistant", "friend and assistant", "chat and productivity"
]

subscription_unlimited_keywords = [
    "unlimited", "unlimited messaging", "unlimited requests", "no limits"
]

login_methods_keywords = ["google", "apple", "facebook", "tiktok", "email", "password"]


## Flatten App Store JSON


In [4]:
ios_rows = []
for item in ios_results:
    row = {
        "source_store": "ios",
        "store_app_id": item.get("appId", ""),
        "numeric_id": item.get("id", ""),
        "title": item.get("title", ""),
        "description": item.get("description", ""),
        "store_url": item.get("url", ""),
        "developer": item.get("developer", ""),
        "developer_website": item.get("developerWebsite", ""),
        "developer_url": item.get("developerUrl", ""),
        "genres": safe_join(item.get("genres")),
        "primary_genre": item.get("primaryGenre", ""),
        "content_rating": item.get("contentRating", ""),
        "languages_store": normalize_lang_codes(item.get("languages")),
        "price": item.get("price", ""),
        "currency": item.get("currency", ""),
        "free": item.get("free", ""),
        "released": item.get("released", ""),
        "updated": item.get("updated", ""),
        "version": item.get("version", ""),
        "score": item.get("score", ""),
        "reviews": item.get("reviews", ""),
    }
    ios_rows.append(row)

ios_df = pd.DataFrame(ios_rows)
ios_df.head()


,source_store,store_app_id,numeric_id,title,description,store_url,developer,developer_website,developer_url,genres,...,content_rating,languages_store,price,currency,free,released,updated,version,score,reviews
0,ios,ai.socialapps.speakmaster,6449190344,PolyBuzz: Chat with Characters,Our app revolutionizes the way we interact wit...,https://apps.apple.com/us/app/polybuzz-chat-wi...,CLOUD WHALE INTERACTIVE TECHNOLOGY LLC.,https://app.polybuzz.ai,https://apps.apple.com/us/developer/cloud-whal...,"Entertainment, Social Networking",...,17+,"ar, ca, cs, da, nl, en, fi, fr, de, hu, id, it...",0.0,USD,True,2023-05-23T07:00:00Z,2026-04-11T10:14:57Z,2.2.2,4.42823,393989
1,ios,app.aiboy,6741568820,FriendX - Virtual AI Boyfriend,Our app is a boyfriend simulator that can be y...,https://apps.apple.com/us/app/friendx-virtual-...,Tequilab LP,https://www.aiboy.app,https://apps.apple.com/us/developer/tequilab-l...,"Entertainment, Lifestyle",...,17+,"en, fr, de, pt, es",0.0,USD,True,2025-02-25T08:00:00Z,2026-03-03T08:58:03Z,5.8,4.71133,724
2,ios,com.tara.ai,6740326134,Talkie Lab - AI Playground,Introducing Talkie Lab — Your Gateway to the L...,https://apps.apple.com/us/app/talkie-lab-ai-pl...,SUBSUP PTE. LTD.,https://talkie-ai.com/,https://apps.apple.com/us/developer/subsup-pte...,Entertainment,...,17+,"ar, en, fi, fr, de, id, it, ja, ko, pt, ru, zh...",0.0,USD,True,2025-02-11T08:00:00Z,2026-04-12T07:12:47Z,2.49.100,4.74786,41382
3,ios,cc.aifriends.soultalk,6504914846,SoulTalk: AI Friends Chat,Step into a world where AI chatbots feel real....,https://apps.apple.com/us/app/soultalk-ai-frie...,SINGULARITY MUTUAL ENTERTAINMENT CULTURE MEDIA...,https://app.starcrowd.ai,https://apps.apple.com/us/developer/singularit...,"Social Networking, Entertainment",...,17+,"en, pt, es",0.0,USD,True,2024-06-28T07:00:00Z,2026-02-04T09:07:53Z,1.9.8,4.52559,25265
4,ios,com.apperry.aibf,1565524138,AI Boyfriend Chat: iBoy,iBoy – Your AI Companion for Support & Self-Di...,https://apps.apple.com/us/app/ai-boyfriend-cha...,LABANE CORP. LTD,https://boyfriend.myanima.ai/,https://apps.apple.com/us/developer/labane-cor...,"Entertainment, Health & Fitness",...,17+,en,0.0,USD,True,2021-05-04T07:00:00Z,2026-04-06T10:54:04Z,2.59.12,4.49969,4809


## Flatten Google Play JSON


In [5]:
android_rows = []
for item in android_results:
    row = {
        "source_store": "android",
        "store_app_id": item.get("appId", ""),
        "numeric_id": "",
        "title": item.get("title", ""),
        "description": item.get("description", ""),
        "store_url": item.get("url", ""),
        "developer": item.get("developer", ""),
        "developer_website": item.get("developerWebsite", ""),
        "developer_url": "",
        "genres": safe_join([c.get("name", "") for c in item.get("categories", [])]) or item.get("genre", ""),
        "primary_genre": item.get("genre", ""),
        "content_rating": item.get("contentRating", ""),
        "languages_store": "",
        "price": item.get("price", ""),
        "currency": item.get("currency", ""),
        "free": item.get("free", ""),
        "released": item.get("released", ""),
        "updated": item.get("updated", ""),
        "version": item.get("version", ""),
        "score": item.get("score", ""),
        "reviews": item.get("reviews", ""),
        "iap_range": item.get("IAPRange", ""),
        "offers_iap": item.get("offersIAP", ""),
        "summary": item.get("summary", ""),
        "developer_email": item.get("developerEmail", ""),
        "privacy_policy": item.get("privacyPolicy", ""),
    }
    android_rows.append(row)

android_df = pd.DataFrame(android_rows)
android_df.head()


,source_store,store_app_id,numeric_id,title,description,store_url,developer,developer_website,developer_url,genres,...,released,updated,version,score,reviews,iap_range,offers_iap,summary,developer_email,privacy_policy
0,android,com.rheality.dot,,dotdotdot Fantasy AI Boyfriend,Fall in love with your perfect AI boyfriend. Y...,https://play.google.com/store/apps/details?id=...,Dotdotdot Inc.,https://dotdotdot.chat,,Entertainment,...,"Apr 3, 2025",1774859152000,1.0.29,3.79,128,$0.99 - $99.99 per item,True,Create your story with AI chat,admin@amore-lab.com,https://dotdotdot-ai.notion.site/Dotdotdot-Pri...
1,android,com.fjsh.alwayssupportiveboyfriend,,AI Boyfriend:Love Chat Partner,Find comfort and support with your virtual boy...,https://play.google.com/store/apps/details?id=...,fjsh,https://www.fjsh.dev/all,,Entertainment,...,"Aug 10, 2024",1756976388000,3.0.24,3.6,35,,False,Your Virtual Boyfriend 24/7,zero@zerotoone.app,https://www.fjsh.dev/all
2,android,ai.boyfriend.virtual.dating.lover.iboy,,Anima: My Virtual AI Boyfriend,Looking for a romantic AI companion who truly ...,https://play.google.com/store/apps/details?id=...,LABANE CORP. Limited,https://boyfriend.myanima.ai/,,Entertainment,...,"Apr 30, 2021",1775808904000,2.59.12,3.982518,988,$0.99 - $199.99 per item,True,Romantic Boyfriend Simulator. Create your Char...,help+aibf@myanima.ai,https://boyfriend.myanima.ai/legal/privacy
3,android,aiboy.companion.app,,AIBoy - Virtual AI Boyfriend,AIBoy – Your Virtual Companion is an AI-powere...,https://play.google.com/store/apps/details?id=...,Life Plus Fitness,https://aiboy.app,,Entertainment,...,"Jan 6, 2026",1775740341000,6.0,4.605769,19,$1.99 - $29.99 per item,True,"Your AI friend for chat, calls, romance &amp;...",support@aiboy.app,http://aiboy.app/privacy
4,android,com.weaver.app.prod,,Talkie: Creative AI Community,Create Your AI-Powered Universe with Talkie — ...,https://play.google.com/store/apps/details?id=...,SUBSUP,https://talkie-ai.com/,,Entertainment,...,"Jun 16, 2023",1775635000000,2.49.102,4.541179,67028,$0.49 - $199.99 per item,True,Unleash Your AI Imagination,feedback@subsup.com,https://talkie-ai.com/static/privacy


## Combining both stores into one full union dataframe


In [6]:
df = pd.concat([ios_df, android_df], ignore_index=True, sort=False)

# Ensure key columns exist
for col in [
    "source_store","store_app_id","numeric_id","title","description","store_url",
    "developer","developer_website","developer_url","genres","primary_genre",
    "content_rating","languages_store","price","currency","free","released",
    "updated","version","score","reviews","iap_range","offers_iap","summary",
    "developer_email","privacy_policy"
]:
    if col not in df.columns:
        df[col] = ""

print("Combined rows:", len(df))
df.head()


Combined rows: 945


,source_store,store_app_id,numeric_id,title,description,store_url,developer,developer_website,developer_url,genres,...,released,updated,version,score,reviews,iap_range,offers_iap,summary,developer_email,privacy_policy
0,ios,ai.socialapps.speakmaster,6449190344,PolyBuzz: Chat with Characters,Our app revolutionizes the way we interact wit...,https://apps.apple.com/us/app/polybuzz-chat-wi...,CLOUD WHALE INTERACTIVE TECHNOLOGY LLC.,https://app.polybuzz.ai,https://apps.apple.com/us/developer/cloud-whal...,"Entertainment, Social Networking",...,2023-05-23T07:00:00Z,2026-04-11T10:14:57Z,2.2.2,4.42823,393989,NaN,NaN,NaN,NaN,NaN
1,ios,app.aiboy,6741568820,FriendX - Virtual AI Boyfriend,Our app is a boyfriend simulator that can be y...,https://apps.apple.com/us/app/friendx-virtual-...,Tequilab LP,https://www.aiboy.app,https://apps.apple.com/us/developer/tequilab-l...,"Entertainment, Lifestyle",...,2025-02-25T08:00:00Z,2026-03-03T08:58:03Z,5.8,4.71133,724,NaN,NaN,NaN,NaN,NaN
2,ios,com.tara.ai,6740326134,Talkie Lab - AI Playground,Introducing Talkie Lab — Your Gateway to the L...,https://apps.apple.com/us/app/talkie-lab-ai-pl...,SUBSUP PTE. LTD.,https://talkie-ai.com/,https://apps.apple.com/us/developer/subsup-pte...,Entertainment,...,2025-02-11T08:00:00Z,2026-04-12T07:12:47Z,2.49.100,4.74786,41382,NaN,NaN,NaN,NaN,NaN
3,ios,cc.aifriends.soultalk,6504914846,SoulTalk: AI Friends Chat,Step into a world where AI chatbots feel real....,https://apps.apple.com/us/app/soultalk-ai-frie...,SINGULARITY MUTUAL ENTERTAINMENT CULTURE MEDIA...,https://app.starcrowd.ai,https://apps.apple.com/us/developer/singularit...,"Social Networking, Entertainment",...,2024-06-28T07:00:00Z,2026-02-04T09:07:53Z,1.9.8,4.52559,25265,NaN,NaN,NaN,NaN,NaN
4,ios,com.apperry.aibf,1565524138,AI Boyfriend Chat: iBoy,iBoy – Your AI Companion for Support & Self-Di...,https://apps.apple.com/us/app/ai-boyfriend-cha...,LABANE CORP. LTD,https://boyfriend.myanima.ai/,https://apps.apple.com/us/developer/labane-cor...,"Entertainment, Health & Fitness",...,2021-05-04T07:00:00Z,2026-04-06T10:54:04Z,2.59.12,4.49969,4809,NaN,NaN,NaN,NaN,NaN


## Adding the required Q1 output columns


In [7]:
required_q1_columns = [
    "app_type",
    "web_accessible",
    "web_url",
    "login_required",
    "login_methods",
    "age_verification_required",
    "age_verification_method",
    "subscription_required_for_long_chat",
    "all_features_available_without_subscription",
    "subscription_features",
    "subscription_cost",
    "languages_supported",
]

for col in required_q1_columns:
    if col not in df.columns:
        df[col] = ""

df.columns.tolist()


['source_store',
 'store_app_id',
 'numeric_id',
 'title',
 'description',
 'store_url',
 'developer',
 'developer_website',
 'developer_url',
 'genres',
 'primary_genre',
 'content_rating',
 'languages_store',
 'price',
 'currency',
 'free',
 'released',
 'updated',
 'version',
 'score',
 'reviews',
 'iap_range',
 'offers_iap',
 'summary',
 'developer_email',
 'privacy_policy',
 'app_type',
 'web_accessible',
 'web_url',
 'login_required',
 'login_methods',
 'age_verification_required',
 'age_verification_method',
 'subscription_required_for_long_chat',
 'all_features_available_without_subscription',
 'subscription_features',
 'subscription_cost',
 'languages_supported']

## Heuristic first-pass filling

These are the draft values inferred from store metadata only and should still be manually verified, especially:
- `web_accessible`
- `login_required`
- `login_methods`
- `age_verification_required`
- `age_verification_method`
- `subscription_required_for_long_chat`


In [8]:
def infer_app_type(title, description, summary=""):
    text = " ".join([str(title or ""), str(description or ""), str(summary or "")]).lower()

    has_companion = contains_any(text, companion_keywords)
    has_general = contains_any(text, general_keywords)
    has_mixed = contains_any(text, mixed_keywords)

    if has_mixed or (has_companion and has_general):
        return "mixed"
    if has_companion:
        return "companion"
    if has_general:
        return "general_purpose"
    return "other"

def infer_web_url(dev_website):
    if isinstance(dev_website, str) and dev_website.strip():
        return dev_website.strip()
    return ""

def infer_web_accessible(title, description, dev_website):
    # Conservative: having a website does not prove interactive web chat.
    # Keep False by default unless obvious from metadata.
    text = " ".join([str(title or ""), str(description or ""), str(dev_website or "")]).lower()
    if not str(dev_website or "").strip():
        return False
    # weak positive if text explicitly points to web app / browser use
    positive = ["web app", "browser", "chat online", "use on the web", "website chat"]
    return any(p in text for p in positive)

def infer_login_required(description):
    text = str(description or "").lower()
    if "no registration required" in text or "without registration" in text:
        return False
    return ""

def infer_login_methods(description):
    text = str(description or "").lower()
    found = [k.title() for k in login_methods_keywords if k in text]
    return ", ".join(dict.fromkeys(found))

def infer_age_verification_required(content_rating):
    rating = str(content_rating or "").lower()
    if "17+" in rating or "mature" in rating or "adult" in rating:
        return True
    return ""

def infer_age_verification_method(content_rating):
    rating = str(content_rating or "").lower()
    if "17+" in rating or "mature" in rating:
        return "Store age rating only; live platform method needs manual verification"
    return ""

def infer_subscription_required_for_long_chat(description, iap_range, offers_iap):
    text = " ".join([str(description or ""), str(iap_range or ""), str(offers_iap or "")]).lower()
    if contains_any(text, subscription_unlimited_keywords):
        return True
    if "premium" in text and ("unlimited" in text or "all content" in text):
        return True
    return ""

def infer_all_features_available_without_subscription(description, offers_iap):
    text = str(description or "").lower()
    if "all content and features" in text and "premium" in text:
        return False
    if offers_iap is False:
        return True
    if str(offers_iap).lower() == "false":
        return True
    return ""

def infer_subscription_features(description, iap_range):
    text = str(description or "")
    low = text.lower()
    features = []
    if "unlimited requests" in low or "unlimited messaging" in low or "unlimited" in low:
        features.append("unlimited messaging/requests")
    if "premium characters" in low:
        features.append("premium characters")
    if "all content and features" in low:
        features.append("all content and features")
    if not features and iap_range:
        features.append("in-app purchases available; exact premium benefits need manual verification")
    return ", ".join(features)

def infer_subscription_cost(price, currency, iap_range, description):
    text = str(description or "")
    # Try explicit weekly/annual in description first
    matches = re.findall(r"(\$\d+(?:\.\d{1,2})?)", text)
    if matches:
        vals = ", ".join(dict.fromkeys(matches[:5]))
        ccy = currency if currency else "USD"
        return f"{ccy}: {vals} (exact plan cadence in description)"
    if iap_range:
        return str(iap_range)
    if price not in ("", None, 0, 0.0):
        return f"{currency} {price}"
    return ""

def infer_languages_supported(source_store, languages_store):
    if source_store == "ios" and str(languages_store).strip():
        return languages_store
    return ""

df["app_type"] = df.apply(lambda r: infer_app_type(r.get("title"), r.get("description"), r.get("summary","")), axis=1)
df["web_url"] = df["developer_website"].apply(infer_web_url)
df["web_accessible"] = df.apply(lambda r: infer_web_accessible(r.get("title"), r.get("description"), r.get("developer_website")), axis=1)
df["login_required"] = df["description"].apply(infer_login_required)
df["login_methods"] = df["description"].apply(infer_login_methods)
df["age_verification_required"] = df["content_rating"].apply(infer_age_verification_required)
df["age_verification_method"] = df["content_rating"].apply(infer_age_verification_method)
df["subscription_required_for_long_chat"] = df.apply(lambda r: infer_subscription_required_for_long_chat(r.get("description"), r.get("iap_range",""), r.get("offers_iap","")), axis=1)
df["all_features_available_without_subscription"] = df.apply(lambda r: infer_all_features_available_without_subscription(r.get("description"), r.get("offers_iap","")), axis=1)
df["subscription_features"] = df.apply(lambda r: infer_subscription_features(r.get("description"), r.get("iap_range","")), axis=1)
df["subscription_cost"] = df.apply(lambda r: infer_subscription_cost(r.get("price"), r.get("currency"), r.get("iap_range",""), r.get("description")), axis=1)
df["languages_supported"] = df.apply(lambda r: infer_languages_supported(r.get("source_store"), r.get("languages_store")), axis=1)

df[["source_store","title","app_type","web_url","web_accessible","login_required","subscription_cost"]].head(15)


,source_store,title,app_type,web_url,web_accessible,login_required,subscription_cost
0,ios,PolyBuzz: Chat with Characters,companion,https://app.polybuzz.ai,False,,nan
1,ios,FriendX - Virtual AI Boyfriend,companion,https://www.aiboy.app,False,,"USD: $7.99, $49.99 (exact plan cadence in desc..."
2,ios,Talkie Lab - AI Playground,general_purpose,https://talkie-ai.com/,False,,nan
3,ios,SoulTalk: AI Friends Chat,companion,https://app.starcrowd.ai,False,,nan
4,ios,AI Boyfriend Chat: iBoy,companion,https://boyfriend.myanima.ai/,False,,nan
5,ios,"Character AI: Chat, Talk, Text",mixed,https://character.ai/,False,,nan
6,ios,AI Boyfriend: Roleplay Chat,companion,,False,,nan
7,ios,Virtual AI Boyfriend: SpicyBF,companion,,False,,nan
8,ios,Ai Boyfriend Chat,companion,,False,,nan
9,ios,CHAI: Social AI Platform- Chat,mixed,https://chai-research.com/,False,,nan


## Building the submission draft

This keeps the core store fields plus the required Q1 fields.


In [9]:
submission_columns = [
    "source_store",
    "store_app_id",
    "title",
    "developer",
    "store_url",
    "developer_website",
    "content_rating",
    "genres",
    "app_type",
    "web_accessible",
    "web_url",
    "login_required",
    "login_methods",
    "age_verification_required",
    "age_verification_method",
    "subscription_required_for_long_chat",
    "all_features_available_without_subscription",
    "subscription_features",
    "subscription_cost",
    "languages_supported",
    "description",
]

submission_columns = [c for c in submission_columns if c in df.columns]
submission_draft = df[submission_columns].copy()

submission_draft.head()


,source_store,store_app_id,title,developer,store_url,developer_website,content_rating,genres,app_type,web_accessible,...,login_required,login_methods,age_verification_required,age_verification_method,subscription_required_for_long_chat,all_features_available_without_subscription,subscription_features,subscription_cost,languages_supported,description
0,ios,ai.socialapps.speakmaster,PolyBuzz: Chat with Characters,CLOUD WHALE INTERACTIVE TECHNOLOGY LLC.,https://apps.apple.com/us/app/polybuzz-chat-wi...,https://app.polybuzz.ai,17+,"Entertainment, Social Networking",companion,False,...,,,True,Store age rating only; live platform method ne...,,,in-app purchases available; exact premium bene...,nan,"ar, ca, cs, da, nl, en, fi, fr, de, hu, id, it...",Our app revolutionizes the way we interact wit...
1,ios,app.aiboy,FriendX - Virtual AI Boyfriend,Tequilab LP,https://apps.apple.com/us/app/friendx-virtual-...,https://www.aiboy.app,17+,"Entertainment, Lifestyle",companion,False,...,,,True,Store age rating only; live platform method ne...,True,False,"unlimited messaging/requests, all content and ...","USD: $7.99, $49.99 (exact plan cadence in desc...","en, fr, de, pt, es",Our app is a boyfriend simulator that can be y...
2,ios,com.tara.ai,Talkie Lab - AI Playground,SUBSUP PTE. LTD.,https://apps.apple.com/us/app/talkie-lab-ai-pl...,https://talkie-ai.com/,17+,Entertainment,general_purpose,False,...,,,True,Store age rating only; live platform method ne...,,,in-app purchases available; exact premium bene...,nan,"ar, en, fi, fr, de, id, it, ja, ko, pt, ru, zh...",Introducing Talkie Lab — Your Gateway to the L...
3,ios,cc.aifriends.soultalk,SoulTalk: AI Friends Chat,SINGULARITY MUTUAL ENTERTAINMENT CULTURE MEDIA...,https://apps.apple.com/us/app/soultalk-ai-frie...,https://app.starcrowd.ai,17+,"Social Networking, Entertainment",companion,False,...,,,True,Store age rating only; live platform method ne...,,,in-app purchases available; exact premium bene...,nan,"en, pt, es",Step into a world where AI chatbots feel real....
4,ios,com.apperry.aibf,AI Boyfriend Chat: iBoy,LABANE CORP. LTD,https://apps.apple.com/us/app/ai-boyfriend-cha...,https://boyfriend.myanima.ai/,17+,"Entertainment, Health & Fitness",companion,False,...,,,True,Store age rating only; live platform method ne...,,,in-app purchases available; exact premium bene...,nan,en,iBoy – Your AI Companion for Support & Self-Di...


## Saving both output files in the current working directory


In [10]:
from pathlib import Path

submission_output_file = Path("q1_submission_draft.csv")
augmented_output_file = Path("q1_augmented_full_union.csv")

submission_draft.to_csv(submission_output_file, index=False)
df.to_csv(augmented_output_file, index=False)

print("Current working directory:", os.getcwd())
print("Saved files:")
print("-", submission_output_file.resolve())
print("-", augmented_output_file.resolve())
print()
print("submission_draft shape:", submission_draft.shape)
print("full augmented df shape:", df.shape)


Current working directory: /Users/naman/Downloads/RouterSense-tasks/Q1)
Saved files:
- /Users/naman/Downloads/RouterSense-tasks/Q1)/q1_submission_draft.csv
- /Users/naman/Downloads/RouterSense-tasks/Q1)/q1_augmented_full_union.csv

submission_draft shape: (945, 21)
full augmented df shape: (945, 38)


## Quick sanity checks


In [11]:
print("App type counts:")
print(df["app_type"].value_counts(dropna=False))
print()

print("Potential companion apps:")
display(df.loc[df["app_type"] == "companion", ["source_store","title","developer","developer_website"]].head(25))


App type counts:
app_type
companion          662
mixed              240
general_purpose     33
other               10
Name: count, dtype: int64

Potential companion apps:


,source_store,title,developer,developer_website
0,ios,PolyBuzz: Chat with Characters,CLOUD WHALE INTERACTIVE TECHNOLOGY LLC.,https://app.polybuzz.ai
1,ios,FriendX - Virtual AI Boyfriend,Tequilab LP,https://www.aiboy.app
3,ios,SoulTalk: AI Friends Chat,SINGULARITY MUTUAL ENTERTAINMENT CULTURE MEDIA...,https://app.starcrowd.ai
4,ios,AI Boyfriend Chat: iBoy,LABANE CORP. LTD,https://boyfriend.myanima.ai/
6,ios,AI Boyfriend: Roleplay Chat,Manuel Worlitzer,
7,ios,Virtual AI Boyfriend: SpicyBF,Onyx Tech Inc,
8,ios,Ai Boyfriend Chat,Alexis Tuil,
10,ios,Him - AI Boyfriend Chat,DOUBLE Y INC.,
11,ios,Soulplay - AI Roleplay,Studio Dopa LLC,https://studiodopa.com/
12,ios,Flipped:Chat with AI Character,SINGAPORE DESERT OASIS TECHNICAL PTE. LIMITED,https://flipped.chat/app-ads.txt
